# Build the 5-minute SPX feature set

This notebook creates the base 5-minute dataset from the minute-level S&P 500 file and prepares it for later feature merging and modeling.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "Data").exists() and (candidate / "Notebooks").exists():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data"
SPX_MINUTE_FILE = DATA_DIR / "^SP500.Last.txt"
ANNUAL_FACTOR = 252 * 78

print(f"Project root: {PROJECT_ROOT}")
print(f"SPX file: {SPX_MINUTE_FILE}")


In [ ]:
# Output columns: bar5 + 5-minute OHLC + target_ann_vol
out_cols = (
    ["bar5"] +
    [f"m{i}_{x}" for i in range(1, 6) for x in ["open", "high", "low", "close"]] +
    ["target_ann_vol"]
)

spx = pd.read_csv(
    SPX_MINUTE_FILE,
    sep=";",
    header=None,
    names=["dt_raw", "open", "high", "low", "close", "volume"],
    usecols=["dt_raw", "open", "high", "low", "close"],
)

spx["dt"] = (
    pd.to_datetime(spx["dt_raw"], format="%Y%m%d %H%M%S", errors="coerce")
      .dt.tz_localize("UTC")
      .dt.tz_convert("America/New_York")
      .dt.tz_localize(None)
)

spx = spx[
    (spx["dt"].dt.time >= pd.Timestamp("09:30").time())
    & (spx["dt"].dt.time <= pd.Timestamp("15:59").time())
].copy()

spx = (
    spx.dropna(subset=["dt"])
       .assign(
           date=lambda x: x["dt"].dt.date,
           bar5=lambda x: x["dt"].dt.floor("5min"),
           minute_no=lambda x: x.groupby(["date", "bar5"]).cumcount() + 1,
       )
)

spx[["open", "high", "low", "close"]] = spx[["open", "high", "low", "close"]].apply(
    pd.to_numeric, errors="coerce"
)

spx = spx.dropna(subset=["close"]).sort_values("dt")
spx["ret_1m"] = spx.groupby("date")["close"].transform(lambda x: np.log(x / x.shift(1)))

target = (
    spx.groupby(["date", "bar5"])["ret_1m"]
       .apply(lambda r: np.sqrt(np.nansum(r**2)) * np.sqrt(ANNUAL_FACTOR))
       .rename("target_ann_vol")
)

ohlc = (
    spx[spx["minute_no"].between(1, 5)]
      .pivot(index=["date", "bar5"], columns="minute_no", values=["open", "high", "low", "close"])
)
# Shift the OHLC features to align with the target being predicted from prior-bar price action.
ohlc = ohlc.groupby(level="date").shift(1)
ohlc.columns = [f"m{m}_{v}" for v, m in ohlc.columns]

rv_5m = (
    ohlc.join(target)
         .reset_index()
         .drop(columns="date")
         .loc[:, out_cols]
         .dropna()
)

print(rv_5m.head())
print(f"\nRows: {len(rv_5m):,}")
print(f"Columns: {len(rv_5m.columns)}")

df5 = rv_5m.copy()
print(df5.head())


In [ ]:
# Merge in any additional engineered feature tables when available
feature_candidates = [
    DATA_DIR / "train_features_5m_clean.parquet",
    DATA_DIR / "processed_sets" / "winsorized_1_99.parquet",
]
feature_files = [p for p in feature_candidates if p.exists()]

if feature_files:
    features_raw = pd.concat([pd.read_parquet(p) for p in feature_files]).sort_index()
    if isinstance(features_raw.index, pd.Index):
        common_times = sorted(set(pd.to_datetime(features_raw.index)) & set(df5["bar5"]))
        features_aligned = features_raw.loc[pd.to_datetime(features_raw.index).isin(common_times)]
        df5_aligned = df5[[c for c in df5.columns if c not in features_raw.columns or c == "bar5"]].copy()
        df5_aligned = df5_aligned[df5_aligned["bar5"].isin(common_times)].set_index("bar5")
        df5 = df5_aligned.join(features_aligned, how="inner").dropna(axis=1).reset_index()
else:
    print("No additional parquet feature tables found; continuing with the base SPX feature set.")

# Remove degenerate zero-vol observations if present
if "target_ann_vol" in df5.columns:
    df5 = df5[df5["target_ann_vol"] != 0].reset_index(drop=True)

print(f"Final dataset size: {df5.shape[0]} rows × {df5.shape[1]} columns")
print(df5[["bar5", "target_ann_vol"]].head())


In [ ]:
OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(exist_ok=True)
df5.to_parquet(OUTPUT_DIR / "train_features_5m_clean.parquet", index=False)
print(f"Saved: {OUTPUT_DIR / 'train_features_5m_clean.parquet'}")


In [ ]:
# Sanity check of the generated data
df5.head()


In [ ]:
# Quick summary print
print(df5.describe(include="all").T.head(10))
